<a href="https://colab.research.google.com/github/routparam12/Python_qns/blob/main/Differentname.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

# Read CSV
df = pd.read_csv("/content/panelists.csv")

# Normalize text (lowercase + handle missing values)
df["firstName"] = df["firstName"].fillna("").str.lower().str.strip()
df["lastName"] = df["lastName"].fillna("").str.lower().str.strip()
df["email"] = df["email"].fillna("").str.lower().str.strip()

# Extract email username (before @)
df["email_name"] = df["email"].str.split("@").str[0]

# Find emails where email username doesn't contain firstname or lastname
mask = ~(
    df.apply(
        lambda row: (
            row["firstName"] in row["email_name"]
            or row["lastName"] in row["email_name"]
        ),
        axis=1
    )
)

# Get suspicious emails
result = df.loc[mask, ["firstName", "lastName", "email"]]

print(result)

/tmp/ipykernel_4749/2502985963.py:4: DtypeWarning: Columns (20,23,32,33) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/panelists.csv")


       firstName    lastName                               email
0          jakub   olszewski  1748613780618_gamekrzywy@gmail.com
1         ronald      martin                heskamran4@gmail.com
4         nathan       lloyd              ngamemasterl@gmail.com
8           alex     martens                 1234zamiz@gmail.com
9       jeanette  sutherland                    jmms47@gmail.com
...          ...         ...                                 ...
123709    joseph      irikwu           dosantomario382@gmail.com
123710     davis       walis         sahorinaoiche1011@gmail.com
123711  eileen f      ashton     johnathanjohnathan735@gmail.com
123716     jason      vargas        tanvirahmedsujon42@gmail.com
123719     tanya      miller            crabtreesally0@gmail.com

[36816 rows x 3 columns]


In [5]:
import pandas as pd

# Read CSV
df = pd.read_csv("/content/panelists.csv")

# Normalize columns
df["firstName"] = df["firstName"].fillna("").str.lower().str.strip()
df["lastName"] = df["lastName"].fillna("").str.lower().str.strip()
df["email"] = df["email"].fillna("").str.lower().str.strip()

# Extract email username (before @)
df["email_name"] = df["email"].str.split("@").str[0]

# Email mismatch condition
email_mask = ~df.apply(
    lambda row: (
        row["firstName"] in row["email_name"]
        or row["lastName"] in row["email_name"]
    ),
    axis=1
)

# Country filter
country_mask = df["countryId"] != 3

# Combine filters
filtered_df = df.loc[
    email_mask & country_mask,
    ["panelistId", "firstName", "lastName", "email"]
]


# Save to CSV
output_file = "filtered_emails2.csv"
filtered_df.to_csv(output_file, index=False)

print(f"Saved {len(filtered_df)} rows to {output_file}")

/tmp/ipykernel_4749/158194380.py:4: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/panelists.csv")


Saved 46213 rows to filtered_emails.csv


In [8]:
import pandas as pd

# Read CSV
df = pd.read_csv("/content/panelists.csv")

# Handle nulls + normalize case
df["email"] = df["email"].fillna("").str.lower()
df["firstName"] = df["firstName"].fillna("").str.lower()
df["lastName"] = df["lastName"].fillna("").str.lower()

# Apply filters
filtered_df = df[
    (df["email"].str.contains("hosen", na=False)) &
    (~df["lastName"].str.contains("hosen", na=False)) &
    (~df["firstName"].str.contains("hosen", na=False)) &
    (df["countryId"] != 3)
][["panelistId", "firstName", "lastName", "email"]]

# Save output
output_file = "filtered_hosen.csv"
filtered_df.to_csv(output_file, index=False)

print(f"Saved {len(filtered_df)} rows to {output_file}")

/tmp/ipykernel_4749/3527784732.py:4: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/panelists.csv")


Saved 21 rows to filtered_hosen.csv


In [ ]:
import pandas as pd
import re
from rapidfuzz import fuzz


def should_remove(first_name, last_name, email):
    username = email.split("@")[0].lower()

    # split names into words
    names = re.findall(r"[a-z]+", f"{first_name} {last_name}".lower())

    for name in names:
        if len(name) < 3:
            continue

        # exact match
        if name in username:
            return True

        # fuzzy match
        if fuzz.partial_ratio(name, username) >= 85:
            return True

    return False


# Load CSV
df = pd.read_csv("input.csv")

# Keep only rows that DON'T match
filtered_df = df[
    ~df.apply(
        lambda x: should_remove(
            x["first_name"],
            x["last_name"],
            x["email"]
        ),
        axis=1
    )
]

filtered_df.to_csv("output.csv", index=False)

print("Done!")